In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/120-million-word-spanish-corpus/spanishText_260000_265000
/kaggle/input/120-million-word-spanish-corpus/spanishText_450000_455000
/kaggle/input/120-million-word-spanish-corpus/spanishText_230000_235000
/kaggle/input/120-million-word-spanish-corpus/spanishText_470000_475000
/kaggle/input/120-million-word-spanish-corpus/spanishText_330000_335000
/kaggle/input/120-million-word-spanish-corpus/spanishText_390000_395000
/kaggle/input/120-million-word-spanish-corpus/spanishText_405000_410000
/kaggle/input/120-million-word-spanish-corpus/spanishText_110000_115000
/kaggle/input/120-million-word-spanish-corpus/spanishText_210000_215000
/kaggle/input/120-million-word-spanish-corpus/spanishText_40000_45000
/kaggle/input/120-million-word-spanish-corpus/spanishText_90000_95000
/kaggle/input/120-million-word-spanish-corpus/spanishText_265000_270000
/kaggle/input/120-million-word-spanish-corpus/spanishText_360000_365000
/kaggle/input/120-million-word-spanish-corpus/spanishText_345000_350

In [2]:
import re
import csv
from pathlib import Path

def extract_spanish_words():
    word_list = []
    target_count = 100000
    word_pattern = re.compile(r'\b[a-záéíóúüñ]{6,}\b', re.IGNORECASE | re.UNICODE)
    file_counter = 0
    
    # Process all files in directory regardless of extension
    data_dir = Path('/kaggle/input/120-million-word-spanish-corpus/')
    
    for file_path in data_dir.glob('*'):  # Match all files
        if not file_path.is_file():
            continue
            
        file_counter += 1
        print(f"Processing file {file_counter}: {file_path.name}")
        
        with open(file_path, 'r', encoding='utf-8', errors='replace') as f:
            content = f.read()
            
            # Extract documents using XML tags
            documents = re.findall(r'<doc\b[^>]*>(.*?)</doc>', content, re.DOTALL)
            
            for doc in documents:
                # Split at the actual ENDOFARTICLE marker
                article_text = doc.split('ENDOFARTICLE.')[0]
                
                # Clean XML tags and special characters
                clean_text = re.sub(r'<[^>]+>', '', article_text)  # Remove XML tags
                clean_text = re.sub(r'[^a-záéíóúüñÁÉÍÓÚÜÑ\s]', ' ', clean_text, flags=re.IGNORECASE)
                
                # Find all matching words
                words = word_pattern.findall(clean_text.lower())
                
                for word in words:
                    if len(word) > 5:
                        word_list.append(word)
                        
                        # Early exit when target reached
                        if len(word_list) >= target_count:
                            with open('spanish_words.csv', 'w', newline='', encoding='latin-1') as csvfile:
                                writer = csv.writer(csvfile)
                                writer.writerow(['Word'])
                                writer.writerows([[w] for w in word_list])
                            print(f"Success! Saved {len(word_list)} words.")
                            return

        print(f"  - Words collected: {len(word_list)}")
        
        if len(word_list) >= target_count:
            break
    
    # Final save if target not reached
    with open('spanish_words.csv', 'w', newline='', encoding='utf-8') as csvfile:
        writer = csv.writer(csvfile)
        writer.writerow(['Word'])
        writer.writerows([[w] for w in word_list])
    print(f"Completed! Saved {len(word_list)} words3.")

# Run the extraction
extract_spanish_words()

Processing file 1: spanishText_260000_265000
Success! Saved 100000 words.


In [3]:
import pandas as pd
df2 = pd.read_csv('spanish_words.csv')
df2.head(5)

,Word
0,cuarto
1,single
2,garbage
3,version
4,lanzado


In [4]:
import csv
import random

def introduce_errors_in_word(word):
    """
    Introduces 1 or 2 errors into a single word by modifying random characters.
    
    If a character is found in the substitutions dictionary, it is replaced with
    its corresponding value. Otherwise, the character is replaced with a randomly
    chosen letter that is different from the original.
    """
    # Dictionary for error substitutions (accent, similar-looking characters, etc.)
    substitutions = {
        'á': 'a', 'é': 'e', 'í': 'i', 'ó': 'o', 'ú': 'u', 'ü': 'u', 'ñ': 'n',
        'l': '1', 'o': '0', 'z': '2', 's': '5', 'B': '8',
        'u': 'v', 'm': 'rn', 'n': 'h', 'c': 'e', 'd': 'cl', 'g': 'q'
    }
    
    if not word:
        return word

    # Decide the number of errors: randomly 1 or 2 (if word length is sufficient)
    num_errors = random.choice([1, 2]) if len(word) >= 2 else 1

    # Select unique indices in the word to modify
    indices = random.sample(range(len(word)), num_errors)
    word_list = list(word)
    
    for idx in indices:
        original_char = word_list[idx]
        if original_char in substitutions:
            word_list[idx] = substitutions[original_char]
        else:
            # For characters without a specific substitution, choose a random letter
            letters = 'abcdefghijklmnopqrstuvwxyz'
            if original_char.isupper():
                letters = letters.upper()
            choices = [c for c in letters if c != original_char]
            word_list[idx] = random.choice(choices)
    
    return ''.join(word_list)

def process_csv(input_csv, output_csv):
    """
    Reads the CSV file 'input_csv', induces errors in the 'Word' column,
    and writes the modified words to 'output_csv'.
    """
    with open(input_csv, 'r', newline='', encoding='utf-8') as infile, \
         open(output_csv, 'w', newline='', encoding='utf-8') as outfile:
        
        reader = csv.DictReader(infile)
        fieldnames = reader.fieldnames
        writer = csv.DictWriter(outfile, fieldnames=fieldnames)
        writer.writeheader()
        
        for row in reader:
            original_word = row['Word']
            row['Word'] = introduce_errors_in_word(original_word)
            writer.writerow(row)

if __name__ == "__main__":
    # Input CSV should have a header 'Word' and one column of words
    input_file = 'spanish_words.csv'
    output_file = 'output_words_with_errors.csv'
    
    process_csv(input_file, output_file)
    print(f"Processed CSV saved to {output_file}")


Processed CSV saved to output_words_with_errors.csv


In [5]:
df = pd.read_csv('output_words_with_errors.csv')
df['Word_correct'] = df2['Word']
df.head(5)

,Word,Word_correct
0,cukrto,cuarto
1,5tngle,single
2,garlaqe,garbage
3,sersi0n,version
4,lgnzado,lanzado


In [6]:
import pandas as pd
from transformers import T5Tokenizer, T5ForConditionalGeneration, Seq2SeqTrainingArguments, Seq2SeqTrainer
from datasets import Dataset

In [7]:
# Convert to Hugging Face Dataset
dataset = Dataset.from_pandas(df)

# Split dataset (adjust ratios as needed)
dataset = dataset.train_test_split(test_size=0.2)

# Initialize tokenizer and model
tokenizer = T5Tokenizer.from_pretrained("vgaraujov/t5-base-spanish")
model = T5ForConditionalGeneration.from_pretrained("vgaraujov/t5-base-spanish")

tokenizer_config.json:   0%|          | 0.00/1.86k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/837k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/1.79k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/588 [00:00<?, ?B/s]

You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565


model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/142 [00:00<?, ?B/s]

In [8]:
# Preprocessing function
def preprocess_function(examples):
    inputs = ["correct: " + word for word in examples["Word"]]
    targets = examples["Word_correct"]
    
    model_inputs = tokenizer(
        inputs,
        max_length=32,
        truncation=True,
        padding="max_length"
    )
    
    labels = tokenizer(
        targets,
        max_length=32,
        truncation=True,
        padding="max_length"
    )
    
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

In [10]:
# Apply preprocessing
tokenized_dataset = dataset.map(
    preprocess_function,
    batched=True,
    remove_columns=dataset["train"].column_names
)

Map:   0%|          | 0/80000 [00:00<?, ? examples/s]

Map:   0%|          | 0/20000 [00:00<?, ? examples/s]

In [11]:
# Training arguments
training_args = Seq2SeqTrainingArguments(
    report_to="none",
    output_dir="./spelling_correction",
    evaluation_strategy="epoch",
    learning_rate=3e-3,
    per_device_train_batch_size=64,
    per_device_eval_batch_size=64,
    weight_decay=0.01,
    save_total_limit=3,
    num_train_epochs=1,
    predict_with_generate=True,
    fp16=False,
    logging_steps=100,
    logging_strategy="steps"
)


/usr/local/lib/python3.10/dist-packages/transformers/training_args.py:1575: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


In [12]:
# Trainer
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["test"],
)

In [ ]:
# wandb login

In [13]:
# Start training
trainer.train()

Passing a tuple of `past_key_values` is deprecated and will be removed in Transformers v4.48.0. You should pass an instance of `EncoderDecoderCache` instead, e.g. `past_key_values=EncoderDecoderCache.from_legacy_cache(past_key_values)`.
/usr/local/lib/python3.10/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


Epoch,Training Loss,Validation Loss
1,0.326100,0.316050


/usr/local/lib/python3.10/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


TrainOutput(global_step=625, training_loss=0.45152759704589845, metrics={'train_runtime': 895.1699, 'train_samples_per_second': 89.369, 'train_steps_per_second': 0.698, 'total_flos': 3423786762240000.0, 'train_loss': 0.45152759704589845, 'epoch': 1.0})

In [14]:
# Save the model
model.save_pretrained("./spelling_correction_final")
tokenizer.save_pretrained("./spelling_correction_final")

('./spelling_correction_final/tokenizer_config.json',
 './spelling_correction_final/special_tokens_map.json',
 './spelling_correction_final/spiece.model',
 './spelling_correction_final/added_tokens.json')

In [16]:
import torch


In [24]:
from transformers import T5ForConditionalGeneration, T5Tokenizer
import torch
import re

class WordLevelCorrector:
    def __init__(self, model_path="./spelling_correction_final"):
        self.tokenizer = T5Tokenizer.from_pretrained(model_path)
        self.model = T5ForConditionalGeneration.from_pretrained(model_path)
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.model.to(self.device)
        
        # Context window configuration
        self.context_window = 3  # Number of previous words to consider
        self.batch_size = 8       # Process multiple words simultaneously

    def correct_text(self, text):
        # Preprocessing with original spacing preservation
        original_words = re.findall(r'\S+|\n', text)
        corrected_words = []
        context_buffer = []

        # Process in batches with context
        for i in range(0, len(original_words), self.batch_size):
            batch = original_words[i:i+self.batch_size]
            
            # Prepare batch inputs with context
            inputs, meta = [], []
            for j, word in enumerate(batch):
                position = i + j
                context_start = max(0, position - self.context_window)
                context = original_words[context_start:position] + corrected_words[context_start:position]
                context = [w for w in context if w != '\n'][-self.context_window:]
                
                inputs.append(f"correct: {' '.join(context)} {word}")
                meta.append({
                    'original': word,
                    'position': position,
                    'is_newline': word == '\n'
                })

            # Batch processing
            encoded = self.tokenizer(
                inputs,
                padding=True,
                truncation=True,
                max_length=32,
                return_tensors="pt"
            ).to(self.device)
            
            outputs = self.model.generate(
                input_ids=encoded.input_ids,
                attention_mask=encoded.attention_mask,
                max_length=32,
                num_beams=3,
                early_stopping=True
            )
            
            # Decode and handle special cases
            decoded = self.tokenizer.batch_decode(outputs, skip_special_tokens=True)
            
            for result, m in zip(decoded, meta):
                if m['is_newline']:
                    corrected_words.append('\n')
                else:
                    # Handle multi-word corrections ("porlamay" → "por la mayor")
                    corrected = result.split()
                    corrected_words.extend(corrected)

        # Reconstruct text with original spacing
        final_text = []
        for w in corrected_words:
            if w == '\n' or not final_text:
                final_text.append(w)
            else:
                final_text.append(' ' + w if not final_text[-1].endswith('\n') else w)
        
        return ''.join(final_text)

# Usage
corrector = WordLevelCorrector()
# text1 = '''el dedo quarto de la mano: luego cu le vendo...'''  # Your full text

corrected = corrector.correct_text(text1)
print(corrected)

schmidt francia antonio francia francia francia schmidt schmidt schmidt descripci
schmidt schmidt schmidt antonio
externos externos sticas sticas sticas descripci antonio descripci construcci
construcci schmidt schmidt schmidt sticas actress sticas sticas
construcci construcci construcci construcci francisco construcci
descripci descripci descripci construcci construcci descripci referencias
actress descripci descripci
externos francia descripci construcci antonio antonio refractarios
descripci descripci construcci construcci refractario refractario refractario refractario
antonio refractario antonio sticas schmidt schmidt schmidt descripci
construcci construcci externos descripci construcci antonio antonio refractario
construcci construcci francia construcci construcci descripci descripci construcci actuaci
francisco francisco schmidt descripci refractario construcci construcci construcci francisco schmidt schmidt
schmidt schmidt schmidt descripci francia descripci construcci construc

In [23]:
# Apply to text1
text1 = '''el dedo quarto de la mano: luego cu le vendo
en el dedo quarto
dos dedos pulgar y indios cubando cada.ctra en fu
do, ir con los dos dedos pulgar y
puchocidado en Lo errarros, pero terter
cixoncito, poniendo mucho cuidado en no errarios
emás tripabic y
espuños poco que concorregir. La decruvo pas
de difútribuye con más ve ocidad, ha de
quiccorregir. La letra pas que e temas trusido
Fe diftribuya comos ve occidad, ha de cftor
le abrosa el comporédor con los que tro dedos,
un Para componer le abriya eicompare los con los que tre
Ima de la mano izquierda y el dedo pulgar rec
palma de la marzo izquierda y el dedo pulgar rec
la mano derecha rue, y la acumoda como ha cu est
mano derecha trae, y
I componedor, ha de citar la vift
ro la letra viene del de la caxa al componedor, ha
que le ha de comar, para tomarla por la ca
letra fíguier te que le ha de comar, para comarla
en la leera figuier se que le ha de comac para comerla
b.c. por la mayor brevedad: porque como «lte
ca,porlamay
viera cola que le a previe en cada vno.
m de tantos tiempos,qualquiera cola que
rincipio le ha de ir
hazc muchos al findel día «Cjerro es que al
síc vay en híbituando: más en
efpicio, hafta que las manos le vay un hibir
con efpicio,haña que
tando habitudas, le hace procurar que con llamano dere
estándico habituadas, fe ha de procur
fin hazer fonecitos, ni
de traiga la letra con frerenidad, y repolo, fin hazer
necefiarios, de que algunos rienen grande vi-
movimientos no neceñarios, de que algunos rien
lo pueden remediar, y no firmen Gno
cio, que quando quieren no lo pueden remediar, y
fin fruto.En teniendo ya las nunos fuelras, y
de goftar el tiempo fin
chas ácftos movimientos, le verá con experiencia
hechas à cftos movimientos, le verá con experi
más que luce la composifición.
'''

# corrected_text = correct_long_text(text1)
# print(corrected_text)

In [26]:
from transformers import T5ForConditionalGeneration, T5Tokenizer
import torch
import re

class UniversalSpellingCorrector:
    def __init__(self, model_name_or_path="vgaraujov/t5-base-spanish"):
        self.tokenizer = T5Tokenizer.from_pretrained(model_name_or_path)
        self.model = T5ForConditionalGeneration.from_pretrained(model_name_or_path)
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.model.to(self.device)
        
        # Auto-detect if base model needs task prefix
        self.requires_prefix = not any([
            "spelling" in model_name_or_path.lower(),
            "correct" in model_name_or_path.lower()
        ])
        
        # Context configuration
        self.context_window = 5  # Words before and after
        self.batch_size = 16

    def _prepare_inputs(self, target_word, context_before, context_after):
        if self.requires_prefix:
            return f"correct: {' '.join(context_before)} [{target_word}] {' '.join(context_after)}"
        return f"{' '.join(context_before)} [{target_word}] {' '.join(context_after)}"

    def correct_text(self, text):
        # Preserve original whitespace
        tokens = re.findall(r'\S+|\n+|\s+', text)
        corrected = []
        
        for i in range(len(tokens)):
            token = tokens[i]
            
            # Preserve non-word tokens
            if not re.match(r'\w+', token):
                corrected.append(token)
                continue
                
            # Get context window
            start = max(0, i - self.context_window)
            end = min(len(tokens), i + self.context_window + 1)
            context = tokens[start:end]
            
            # Mark target word
            context_str = ' '.join([
                t if j != i - start else f"[{t}]" 
                for j, t in enumerate(context)
            ])
            
            # Format input
            input_text = f"correct: {context_str}" if self.requires_prefix else context_str
            
            # Generate correction
            inputs = self.tokenizer(
                input_text,
                return_tensors="pt",
                max_length=128,
                truncation=True
            ).to(self.device)
            
            outputs = self.model.generate(
                inputs.input_ids,
                attention_mask=inputs.attention_mask,
                max_length=32,
                num_beams=3,
                early_stopping=True
            )
            
            decoded = self.tokenizer.decode(
                outputs[0],
                skip_special_tokens=True
            )
            
            # Extract correction (handle multi-word outputs)
            correction = decoded.split(']')[0].strip()
            corrected.append(correction if correction else token)

        # Reconstruct text with original spacing
        return ''.join(corrected)

# Usage examples
if __name__ == "__main__":
    # For base model (weak performance)
    base_corrector = UniversalSpellingCorrector("vgaraujov/t5-base-spanish")
    
    # For fine-tuned model
    tuned_corrector = UniversalSpellingCorrector("./spelling_correction_final")
    
    text = "cuadco ver5son qarbage"
    
    print("Base model correction:", base_corrector.correct_text(text))
    print("Fine-tuned correction:", tuned_corrector.correct_text(text))

Base model correction: arararararararsararbagear ararsarar: cuad::::::::::::: a...q............::::ar......sc...
Fine-tuned correction: schmidt schmidt schmidt


In [ ]:
# import editdistance
# import numpy as np
# from sklearn.feature_extraction.text import CountVectorizer
# from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction


# def calculate_cer(text1, text2):
#     """Calculate Character Error Rate (CER) between two texts."""
#     text1 = text1.replace(" ", "").replace("\n", "")
#     text2 = text2.replace(" ", "").replace("\n", "")
    
#     # Calculate edit distance and CER
#     distance = editdistance.eval(text1, text2)
#     cer = distance / max(len(text1), len(text2))
    
#     return cer


# def calculate_wer(text1, text2):
#     """Calculate Word Error Rate (WER) between two texts."""
#     words1 = text1.split()
#     words2 = text2.split()
    
#     # Calculate edit distance and WER
#     distance = editdistance.eval(words1, words2)
#     wer = distance / max(len(words1), len(words2))
    
#     return wer


# def calculate_cosine_similarity(text1, text2):
#     """Calculate Cosine Similarity between two texts."""
#     vectorizer = CountVectorizer().fit_transform([text1, text2])
#     vectors = vectorizer.toarray()
#     cos_sim = np.dot(vectors[0], vectors[1]) / (np.linalg.norm(vectors[0]) * np.linalg.norm(vectors[1]))
#     return cos_sim


# def calculate_bleu(text1, text2):
#     """Calculate BLEU score between two texts."""
#     reference = [text2.split()]
#     candidate = text1.split()
#     smoothing_function = SmoothingFunction().method1
#     return sentence_bleu(reference, candidate, smoothing_function=smoothing_function)


# # Example usage
# text2 = '''en el dedo quarto de la mano: luego en lyeyendo lo que ha toma-
# do, ir con los dos dedos pulgar, y indice echando cada etra en su
# caxoncito, poniendo mucho cuidado en no errarlos, para tener
# después poco que corregir. La letra, para que este más tratable, y
# se distribuya con más velocidad, ha de estar siempre mojada.
# Para componer se abraza el componedor con los quatro dedos,
# y palma de la mano izquierda, y el dedo pulgar recibe la letra que
# la mano derecha trae, y la acomoda como ha de estar: y en qua-
# to la letra viene desde la caxa al componedor, ha de estar la vista
# en la letra siguiente te que se ha de tomar, para tomarla por la ca-
# beza, por la mayor brevedad: porque como este exercicio cons
# ta de tanto tiempos, qualquiera cosa que se abrevie en cada vno
# haze mucho al fin del dia. Cierto es, que al principio se ha de ir
# con espacio, hasta que las manos se vayan habituando: mas en
# estando habituadas, se hace procurar que con la mano derecha
# se traiga la letra con serenidad, y reposo, sin hacer sonecitos, ni
# movimientos no necessarios, de que algunos tienen grande vi-
# cio, que quando quieren no lo pueden remediar, y no serven sino
# de gastar el tiempo sin fruto. En teniendo ya las manos sueltas, y
# hechas á estos movimientos, se verá con experiencia lo mucho
# mas que luce la composicion.'''

# print(f"CER: {calculate_cer(text1, text2):.4f}")
# print(f"WER: {calculate_wer(text1, text2):.4f}")
# print(f"Cosine Similarity: {calculate_cosine_similarity(text1, text2):.4f}")
# print(f"BLEU Score: {calculate_bleu(text1, text2):.4f}")



In [ ]:
# def calculate_jaccard_similarity(text1, text2):
#     """Calculate Jaccard Similarity between two texts."""
#     set1 = set(text1.split())
#     set2 = set(text2.split())
#     intersection = len(set1.intersection(set2))
#     union = len(set1.union(set2))
#     return intersection / union if union != 0 else 0.0


# print(f"Jaccard Similarity: {calculate_jaccard_similarity(text1, text2):.4f}")
